# Stage 5: 多分辨率 Leiden 聚类 + 显式遍历

以多个 Leiden 分辨率运行聚类并比较，使 PI 能选择最符合生物学粒度的分辨率。

**默认**：在 stage 4 选定的嵌入上做多分辨率 Leiden。

**扩展槽**（注释掉的 cell）：任何写出 `obs["{method}_clusters"]` 的聚类方法
均可与 `leiden_res_*` 列共存。ACDC **不是**默认依赖，安装后取消注释即可插入。

**本 notebook 产出**：
- `obs["leiden_res_{resolution}"]` 列
- 各分辨率着色的 UMAP 图
- 显式 for 循环产出的聚类指标对比表
- Stage 5 checkpoint h5ad，供 stage 6 标注使用

## 如何回跑（迭代调参机制）

### 管线位置
- **上游**：Stage 4（多方法嵌入），读 `stage4_embedded_v*.h5ad`
- **下游**：Stage 6（多方法注释），产出 `stage5_clustered_v*.h5ad`

### 为什么要迭代回跑？
聚类分群的分辨率选择直接影响下游注释的质量。如果在 stage 6（注释时发现
某个细胞类型被错误拆分或合并、LLM 判决分歧高）发现问题，可能需要：
- 调整 `RESOLUTIONS` 列表（加更细或更粗的分辨率）
- 换用不同的嵌入（改 `USE_REP` 指向 stage 4 的另一个嵌入）
- 换用 stage 4 的另一个版本（不同嵌入方法/参数组合）

PI 的原始要求（来自项目构思）：
> "注释这一步，也可能在注释的过程中发现前面高可变基因的选择、embedding 的构建，
> 还有分群的参数等等需要调整，应可以随时调回去重跑一些流程，需要建立这种循环不断迭代的机制。"

### 如何回跑（三步操作）
1. **改 `UPSTREAM_PATH`**——指向要复用的上游文件版本
   （例如 `nancang_stage4_embedded_v2.h5ad`）
2. **改 `OUTPUT_PATH`**——bump 版本号 `_v1` → `_v2`
   （例如 `nancang_stage5_clustered_v2.h5ad`）
3. **调整参数**（在下方 `# === PARAMS ===` 区域改 `RESOLUTIONS` 或 `USE_REP`）
   → 重跑本 notebook（Cell → Run All）

### 版本约定
- **`_v1` / `_v2` / ...**：每次调参重跑 bump 一位版本号。
  旧版 `.h5ad` 文件**不覆盖不删除**，保留在 `results/` 目录供追溯对比。
- **`experimental`**：刚跑出、尚未经 PI 审查确认的版本（默认值）。
- **`promoted`**：PI 审查后认为分群质量可接受、可传给下游使用的正式版本。
  PI 在 Jupyter 中打开 `.h5ad` 后手动改 `adata.uns["status"] = "promoted"` 再保存。
- **下游取数**：stage 6 的 `UPSTREAM_PATH` 指向你决定采用的 stage 5 版本即可。

### 追溯链（自动写入 h5ad 的 `adata.uns`）
本 notebook 在写出前自动记录以下字段，供后续审计查询：
- `stage` = `"stage5_clustered"`（本 stage 标识）
- `status` = `"experimental"`（PI 审查后改为 `"promoted"`）
- `upstream` = 本次读入的上游文件路径列表
- `version` = 与 `OUTPUT_PATH` 一致的版本号（`"v1"` / `"v2"` / ...）

如需查询"stage 5 有哪些版本？哪些依赖 stage4_v1？"，
可直接在 Python 中 glob `results/` 目录检查每个 `.h5ad` 的 `adata.uns`。

In [ ]:
# === PARAMS ===
# UPSTREAM_PATH  — stage 4 产出文件路径。
#                   如需回跑：指向要复用的上游版本（如 stage4_embedded_v2.h5ad）。
# OUTPUT_PATH    — 本 stage 产出 checkpoint 路径。
#                   版本号 _v1 与 adata.uns["version"] 保持一致。
#                   如需回跑：bump 版本号 _v1->v2，旧版不覆盖。
# USE_REP        — 用于构建邻居图的嵌入键名（obsm 中的 key）。
#                   如需换用不同嵌入：改为 stage 4 产出的其他嵌入，
#                   如 "X_pca_harmony"、"X_scVI"、"X_pca"。
# RESOLUTIONS    — 要尝试的 Leiden 分辨率列表。
#                   粗分辨率（0.2-0.4）捕获大谱系；
#                   细分辨率（1.0+）区分亚型。
# RANDOM_SEED    — 固定随机种子，保证可复现。

UPSTREAM_PATH = "results/nancang_stage4_embedded_v1.h5ad"
OUTPUT_PATH   = "results/nancang_stage5_clustered_v1.h5ad"

USE_REP      = "X_scVI"      # 邻居图基于此嵌入；换成 stage 4 其他嵌入需 bump OUTPUT_PATH 版本号
RESOLUTIONS  = [0.2, 0.4, 0.6, 0.8, 1.0, 1.2, 1.5, 2.0]
RANDOM_SEED  = 42

In [ ]:
# 确保框架 src/ 在 sys.path 上，CWD 为项目根目录。
import sys, os
_root = os.getcwd()
if not os.path.isdir(os.path.join(_root, "src", "scrna_integration")):
    _root = os.path.abspath(os.path.join(_root, ".."))
if os.path.join(_root, "src") not in sys.path:
    sys.path.insert(0, os.path.join(_root, "src"))
os.chdir(_root)
os.makedirs("results/figures", exist_ok=True)
os.makedirs("results/figures/sweep_stage5", exist_ok=True)
print(f"PROJECT_ROOT: {_root}")

In [ ]:
# 导入（scanpy 原生 API + 框架函数）。
import scanpy as sc
import scipy.sparse as sp
import numpy as np
import matplotlib.pyplot as plt
import datetime
import warnings

# 直接从 scorers 模块导入——无回调，在 for 循环中直接调用
from scrna_integration.scorers import clustering_metrics

sc.settings.verbosity = 2
sc.settings.set_figure_params(dpi=100, facecolor="white", frameon=False)

print("加载上游:", UPSTREAM_PATH)
adata = sc.read_h5ad(UPSTREAM_PATH)
print(f"已加载: {adata.n_obs:,} 细胞 x {adata.n_vars:,} 基因")
print(f"X dtype: {adata.X.dtype}  |  sparse: {sp.issparse(adata.X)}")
print(f"obsm keys: {list(adata.obsm.keys())}")
print(f"use_rep='{USE_REP}' -- 存在: {USE_REP in adata.obsm}")

## 计算邻居图

在 stage 4 选定嵌入（`USE_REP`）上构建 k 近邻图。
所有 Leiden 分辨率共享该图——只算一次。
**为什么共享邻居图？** 不同分辨率只改变聚类的粒度，不需要
每次重新计算细胞间距离关系。共享避免了重复计算和内存浪费。

In [ ]:
# 在选定的嵌入上计算邻居图。
# n_pcs=None 使用嵌入的全部维度（嵌入已是低维，无需再做 PCA 降维）。
print(f"\n===== Neighbors on {USE_REP} =====")
sc.pp.neighbors(
    adata,
    use_rep=USE_REP,
    n_pcs=None,
    random_state=RANDOM_SEED,
)
print(f"Neighbor graph: {adata.obsp['connectivities'].shape}")
print(f"  n_neighbors={adata.uns['neighbors']['params']['n_neighbors']}")

## 多分辨率 Leiden 聚类

每个分辨率各做一次 Leiden 聚类。所有 `leiden_res_*` 列共存以供对比。
使用 `flavor="igraph"` 以兼容 scanpy >= 1.10。
**为什么做多个分辨率？** 没有"正确"的分辨率——不同生物学问题需要
不同粒度。粗分辨率（0.2-0.4）捕获大细胞谱系；细分辨率（1.0+）
区分亚型。PI 看完结果后根据生物学背景选择最合适的分辨率。

In [ ]:
# 多分辨率 Leiden 聚类：每个分辨率一个 obs 列。
print(f"\n===== Leiden: {len(RESOLUTIONS)} resolutions =====")

for res in RESOLUTIONS:
    key = f"leiden_res_{res}"
    print(f"  resolution={res} -> obs['{key}']")
    sc.tl.leiden(
        adata,
        resolution=res,
        key_added=key,
        flavor="igraph",
        random_state=RANDOM_SEED,
    )
    n_clusters = adata.obs[key].nunique()
    print(f"    clusters: {n_clusters}")

leiden_columns = [c for c in adata.obs.columns if c.startswith("leiden_res_")]
print(f"\nLeiden columns produced: {leiden_columns}")

## 显式遍历分辨率 + 聚类指标

直接遍历分辨率列表，对每个分辨率：
1. 拷贝 adata
2. 在该分辨率运行 Leiden 聚类
3. 直接调用 `clustering_metrics(adata_copy, cluster_key=key)` 获取指标
4. 收集到 DataFrame

**没有回调、没有 `sweep()`**——学生逐行可读。

指标（数据允许时计算）：
- **silhouette**——PCA 空间上的簇分离度
- **ari**——与已知标签的调整兰德指数

对比表写入 `results/figures/sweep_stage5/sweep_report.md`。

In [ ]:
# 显式 for 循环：遍历各分辨率，运行 Leiden 并计算聚类指标。
# 每个分辨率在独立拷贝上运行，避免列名冲突。
print("\n===== 显式遍历分辨率 + clustering_metrics =====\n")

import pandas as pd

results = []
for res in RESOLUTIONS:
    print(f"--- resolution={res} ---")

    # 拷贝 AnnData，在该分辨率运行 Leiden
    adata_copy = adata.copy()
    key = f"leiden_res_{res}"
    sc.tl.leiden(
        adata_copy,
        resolution=res,
        key_added=key,
        flavor="igraph",
        random_state=RANDOM_SEED,
    )

    # 直接调用聚类指标函数——显式传入 cluster_key=key 避免 auto-detect 误选其他 leiden 列
    m = clustering_metrics(adata_copy, cluster_key=key)
    results.append({"resolution": res, **m})

    n_cl = adata_copy.obs[key].nunique()
    metrics_str = ", ".join(f"{k}={v:.4f}" for k, v in m.items()
                            if isinstance(v, float) and not np.isnan(v))
    print(f"  簇数: {n_cl}  指标: {metrics_str}")

# 收集为 DataFrame 对比表
sweep_df = pd.DataFrame(results)
os.makedirs("results/figures/sweep_stage5", exist_ok=True)

# 写 Markdown 报告
lines = ["# Stage 5 聚类对比报告\n",
         f"**{len(RESOLUTIONS)} 个分辨率** 已评估。\n",
         "## 指标表\n"]
lines.append("| " + " | ".join(sweep_df.columns) + " |")
lines.append("|" + "|".join(" --- " for _ in sweep_df.columns) + "|")
for _, row in sweep_df.iterrows():
    vals = []
    for col in sweep_df.columns:
        v = row[col]
        if isinstance(v, float):
            vals.append(f"{v:.4f}" if not np.isnan(v) else "N/A")
        else:
            vals.append(str(v))
    lines.append("| " + " | ".join(vals) + " |")
with open("results/figures/sweep_stage5/sweep_report.md", "w") as f:
    f.write("\n".join(lines) + "\n")

# 展示对比表
print("\n聚类指标对比表:")
try:
    from IPython.display import display as ipy_display
    ipy_display(sweep_df)
except ImportError:
    print(sweep_df)

print("\n对比报告: results/figures/sweep_stage5/sweep_report.md")
adata.uns["stage5_sweep_v1"] = {
    "resolutions_swept": RESOLUTIONS,
    "scorer": "clustering_metrics",
    "report_dir": "results/figures/sweep_stage5",
    "timestamp": datetime.datetime.now().isoformat(),
}

## （可选）其他聚类方法——扩展槽

任何写出 `obs["{method}_clusters"]` 的聚类方法均可与 `leiden_res_*` 列共存
并以同样方式对比。

**ACDC**（已注释）：全局搜索最优分区。**不**是默认依赖（在 GCPL 数据上
运行太慢）。PI 安装后取消注释，新列自动进入 stage 6。

**新方法的通用模式**：写 `obs["{method}_clusters"]` = labels，
然后加入显式 for 循环的 candidates 列表或单独对比——
与 stage 4 嵌入同样的并行槽位约定，零框架改动。

In [ ]:
# # === ACDC clustering (commented out -- NOT a default dependency) ===
# # PREREQUISITE: pip install acdc_py
# # Enable by removing comments below.
#
# # import acdc_py  # 包名/导入名以 PyPI 实际为准，启用前先确认；ACDC 非默认依赖
# # # ACDC searches for an optimal partition.
# # acdc_result = ACDC.ACDC(adata, ...)
# # adata.obs["acdc_clusters"] = acdc_result.labels
# # print(f"ACDC: {adata.obs['acdc_clusters'].nunique()} clusters")
#
# print("ACDC cell is commented out. "
#       "Uncomment when acdc_py is installed and suitable for this dataset.")
#
# # === Add any future method here ===
# # Pattern: write adata.obs["{method}_clusters"] = labels
# # Then add to sweep candidates or compare standalone.
# # Example: adata.obs["foocluster_clusters"] = foo_cluster.fit_predict(
# #     adata.obsm[USE_REP])

## 可视化——各分辨率着色的 UMAP

按每个 `leiden_res_*` 列的聚类标签着色 UMAP。
PI 对比选择最符合生物学结构的分辨率。

In [ ]:
# 按每个 leiden_res_* 列的聚类标签着色 UMAP。
print("\n===== UMAP per resolution =====")

leiden_cols = sorted(
    [c for c in adata.obs.columns if c.startswith("leiden_res_")],
    key=lambda x: float(x.split("_")[-1]),
)

# 从已有邻居图计算 UMAP。
sc.tl.umap(adata, random_state=RANDOM_SEED)

for col in leiden_cols:
    n_clusters = adata.obs[col].nunique()
    print(f"  {col}: {n_clusters} clusters")

    sc.pl.umap(
        adata, color=col,
        title=f"Leiden res={col.split('_')[-1]} ({n_clusters} clusters)",
        legend_loc="on data" if n_clusters <= 10 else "right margin",
        frameon=False,
        save=f"_stage5_{col}.png",
    )
    # Move from scanpy default figures/ dir to results/figures/.
    src = f"figures/umap_stage5_{col}.png"
    dst = f"results/figures/stage5_umap_{col}.png"
    if os.path.exists(src):
        os.rename(src, dst)
        print(f"    Saved {dst}")
    plt.close("all")

print(f"\nUMAP plots saved for {len(leiden_cols)} resolutions.")

## 运行元数据——plain `adata.uns` 写入

版本化键记录聚类参数用于溯源。

In [ ]:
# 记录聚类运行元数据。
print("\n===== Run metadata =====")

adata.uns["leiden_v1"] = {
    "use_rep": USE_REP,
    "resolutions": RESOLUTIONS,
    "flavor": "igraph",
    "n_neighbors": adata.uns["neighbors"]["params"]["n_neighbors"],
    "timestamp": datetime.datetime.now().isoformat(),
}

# 统一追踪字段——stage + version（与上游字段合并，保持 stage3-7 命名一致）
adata.uns["stage"] = "stage5_clustered"     # 本 stage 标识
adata.uns["version"] = "v1"                  # 与 OUTPUT_PATH 版本号一致
adata.uns["upstream"] = [UPSTREAM_PATH]
adata.uns["status"] = "experimental"

# 各分辨率的簇数汇总。
cluster_summary = {}
for col in leiden_cols:
    cluster_summary[col] = int(adata.obs[col].nunique())
adata.uns["leiden_v1"]["cluster_counts"] = cluster_summary

print("Clusters per resolution:")
for col, n in cluster_summary.items():
    print(f"  {col}: {n}")
print(f"status: {adata.uns['status']}")

In [ ]:
# 内存纪律自检——写入前一次断言。
# 守卫最高影响的内存退化。
import scipy.sparse as sp
import numpy as np
assert sp.issparse(adata.X) and adata.X.dtype == np.float32, (
    f"adata.X invariants violated: sparse={sp.issparse(adata.X)}, dtype={adata.X.dtype}"
)
print("Memory self-check passed: X is sparse CSR float32.")

In [ ]:
# 将本 stage 产出写出到磁盘（内存纪律 #4：lzf 压缩）。
adata.write_h5ad(OUTPUT_PATH, compression="lzf")
print(f"Wrote {OUTPUT_PATH}")

import os
assert os.path.exists(OUTPUT_PATH), f"Output NOT found: {OUTPUT_PATH}"
print(f"Verified: {OUTPUT_PATH} ({os.path.getsize(OUTPUT_PATH):,} bytes)")

In [ ]:
# 跨 stage 边界释放内存（内存纪律 #3）。
del adata
import gc
gc.collect()
print("Memory released.")